# Traffic Monitoring Pipeline (RT-DETR + ByteTrack + Temporal VAE/Rules)
![image.png](./image.png)
### Streamlined Notebook Layout

This notebook follows the same end-to-end flow as the reference pipeline diagram:
1. RT-DETR detection
2. ByteTrack tracking
3. Tracklet feature extraction
4. Temporal VAE anomaly scoring
5. Fused decision
6. Alerts and logging

Execution order:
- Cell 2: RT-DETR + tracker + tracklet features
- Cell 5a.5: bootstrap defaults for downstream VAE cells
- Cell 5b: synthetic wrong-way sample generation
- Cell 5c: VAE training on normal + synthetic wrong-way data
- Cell 5d: VAE threshold calibration
- Cell 7: fused decision overlay + alert log export
- Cell 10: VAE comparison view
- Cell 10b: VAE threshold and input diagnostics

In [16]:
# Notebook Configuration
# Edit values here first when you want to tweak the pipeline.

# --- RT-DETR / tracking ---
video_source = "CCTV_Sleman.mp4"
model_path = "rtdetr-l.pt"
tracker_cfg = "rtdetr_tuned.yaml"
target_classes = [2, 3, 5, 7]  # Cars, motorcycles, buses, trucks
class_name_map = {2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}
ENABLE_PICKUP_TO_CAR_REMAP = True
PICKUP_REMAP_MAX_TRUCK_CONF = 0.72
PICKUP_REMAP_MAX_AREA_RATIO = 0.055
PICKUP_REMAP_MAX_HEIGHT_RATIO = 0.24
MOVEMENT_THRESHOLD = 5
MAX_MISSED_FRAMES = 45
TRACKLET_HISTORY = 30
OUTPUT_FILENAME = "11m_monitoring_output.mp4"

# --- VAE / GANomaly ---
VAE_NORMAL_SOURCE_LEFT = "vae_split_samples/left"
VAE_NORMAL_SOURCE_RIGHT = "vae_split_samples/right"
VAE_TEST_SOURCE = "CCTV_Sleman.mp4"
VAE_WEIGHTS_LEFT_PATH = "temporal_vae_left_weights.pt"
VAE_WEIGHTS_RIGHT_PATH = "temporal_vae_right_weights.pt"
VAE_IMG_SIZE = 64
VAE_SEQ_LEN = 16
VAE_FRAME_STEP = 2
VAE_SEQ_STRIDE = 2
VAE_MAX_FRAMES_PER_VIDEO = 2500
VAE_MAX_SEQUENCES = 1800
VAE_BATCH_SIZE = 8
VAE_LATENT_DIM = 64
VAE_HIDDEN_DIM = 128
VAE_EMB_DIM = 128
VAE_EXTRACT_TRACKLETS = True
VAE_TRACKLET_OUT_LEFT = "vae_tracklet_features_left.csv"
VAE_TRACKLET_OUT_RIGHT = "vae_tracklet_features_right.csv"
VAE_TRACKLET_DETECTOR = "yolo11m.pt"
VAE_TRACKLET_TRACKER_CFG = "bytetrack_tuned.yaml"
VAE_TRAIN_IF_MISSING = True
VAE_TRAIN_EPOCHS = 5
VAE_TRAIN_LR = 1e-3
VAE_TRAIN_KL_WEIGHT = 0.05
VAE_DIVIDER_BUFFER = 0
KL_WEIGHT = 0.05
PERCENTILE_LEFT = 98.5
PERCENTILE_RIGHT = 98.5
OUTPUT_VIDEO = "vae_gan_bytetrack_anomaly_overlay.mp4"
OUTPUT_CSV = "vae_gan_fused_alerts.csv"


In [18]:
# Combined Object Counting and Sidewalk Violation Detection
import cv2
import torch
from ultralytics import YOLO, solutions
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time
from collections import defaultdict, deque

# --- 1. SETUP & CONFIGURATION ---

# Check GPU
print(f"CUDA status: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Current VRAM Allocated: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")

# Define file paths and settings
video_source = "CCTV_Sleman.mp4"
model_path = "rtdetr-l.pt"
tracker_cfg = "rtdetr_tuned.yaml"
target_classes = [2, 3, 5, 7]  # Cars(2), Motorcycles(3), Buses(5), Trucks(7)
class_name_map = {2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}

# Scene-specific relabel: treat small pickup-like truck detections as car.
ENABLE_PICKUP_TO_CAR_REMAP = True
PICKUP_REMAP_MAX_TRUCK_CONF = 0.72
PICKUP_REMAP_MAX_AREA_RATIO = 0.055
PICKUP_REMAP_MAX_HEIGHT_RATIO = 0.24

# Minimum pixel movement per frame to be considered "moving" (avoids flagging stopped vehicles)
MOVEMENT_THRESHOLD = 5
MAX_MISSED_FRAMES = 45

# --- 2. REGION DEFINITIONS ---

# Initialize video capture to get dimensions
cap = cv2.VideoCapture(video_source)
assert cap.isOpened(), "Error reading video file"
w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))

# Line Counting Region (Horizontal line)
line_points = [(0, int(h / 1.55)), (w, int(h / 1.55))]

# Sidewalk Zones (Polygons) - pre-converted to numpy for efficiency
sidewalk_zones_np = [
    np.array([(48, 1041), (755, 251), (788, 253), (160, 1041)], np.int32),   # LEFT SIDEWALK
    np.array([(1266, 277), (1919, 961), (1919, 839), (1318, 281)], np.int32), # RIGHT SIDEWALK
]

# Divider-based guard so road traffic near the midline is ignored
road_divider_x = int(w / 1.88)
divider_buffer_px = 80  # widen or shrink if mid-road false positives persist

# --- 3. INITIALIZATION ---

# Output video writer
output_filename = "11m_monitoring_output.mp4"
video_writer = cv2.VideoWriter(output_filename, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

# Initialize ObjectCounter for the Line (handles tracking and counting)
counter = solutions.ObjectCounter(
    model=model_path,
    region=line_points,
    classes=target_classes,
    tracker=tracker_cfg,
    show=False,
    line_width=2,
 )

def init_kalman(x, y):
    kf = cv2.KalmanFilter(4, 2)
    kf.transitionMatrix = np.array([[1, 0, 1, 0],
                                    [0, 1, 0, 1],
                                    [0, 0, 1, 0],
                                    [0, 0, 0, 1]], dtype=np.float32)
    kf.measurementMatrix = np.array([[1, 0, 0, 0],
                                     [0, 1, 0, 0]], dtype=np.float32)
    kf.processNoiseCov = np.eye(4, dtype=np.float32) * 1e-2
    kf.measurementNoiseCov = np.eye(2, dtype=np.float32) * 1e-1
    kf.errorCovPost = np.eye(4, dtype=np.float32)
    kf.statePost = np.array([[x], [y], [0], [0]], dtype=np.float32)
    return kf

def remap_vehicle_class(raw_cls, box, frame_w, frame_h, conf=None):
    cls_id = int(raw_cls)
    if (not ENABLE_PICKUP_TO_CAR_REMAP) or cls_id != 7:
        return cls_id

    x1, y1, x2, y2 = [float(v) for v in box]
    bw = max(1.0, x2 - x1)
    bh = max(1.0, y2 - y1)
    area_ratio = (bw * bh) / float(max(frame_w * frame_h, 1))
    h_ratio = bh / float(max(frame_h, 1))

    conf_ok = (conf is None) or (float(conf) <= PICKUP_REMAP_MAX_TRUCK_CONF)
    size_ok = (area_ratio <= PICKUP_REMAP_MAX_AREA_RATIO) and (h_ratio <= PICKUP_REMAP_MAX_HEIGHT_RATIO)

    if conf_ok and size_ok:
        return 2
    return cls_id

# Initialize variables for Sidewalk violations
left_sidewalk_violations = 0   # Left zone: wrong direction = moving DOWN (y increasing)
right_sidewalk_violations = 0  # Right zone: wrong direction = moving UP (y decreasing)
violation_ids = set()

# Track previous centroid positions to determine movement direction
# { track_id: (x, y) }
prev_positions = {}

# Tracklet features (history + aggregates)
TRACKLET_HISTORY = 30  # number of points kept per active track
tracklet_history = defaultdict(lambda: deque(maxlen=TRACKLET_HISTORY))
tracklet_distance = defaultdict(float)   # cumulative travel distance in pixels
tracklet_conf_sum = defaultdict(float)   # sum of confidences for avg confidence
tracklet_frames_seen = defaultdict(int)
tracklet_first_last = {}                 # {track_id: [first_frame, last_frame]}
tracklet_class_votes = defaultdict(lambda: defaultdict(int))
tracklet_speed_values = defaultdict(list)       # per-step speed samples (px/s)
tracklet_accel_values = defaultdict(list)       # per-step acceleration samples (px/s^2)
tracklet_prev_speed = {}                        # previous speed for acceleration calc
tracklet_step_distance_values = defaultdict(list)  # per-step distance samples (px)
tracklet_kf = {}  # Kalman filter per track id
tracklet_missed_frames = defaultdict(int)

frame_count = 0
fps_smooth = None
print(f"Starting combined monitoring on {video_source}...")
print(f"Left zone violation = moving DOWN | Right zone violation = moving UP")
print(f"Tracker config: {tracker_cfg} | missed frame grace: {MAX_MISSED_FRAMES}")

# --- 4. MAIN PROCESSING LOOP ---
while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break
    
    frame_start_time = time.time()
    frame_count += 1
    
    # Process the frame with the Line Counter
    result = counter(frame)
    im0 = result.plot_im 

    # Draw the sidewalk zones once per frame (outside the per-object loop)
    overlay = im0.copy()
    cv2.fillPoly(overlay, sidewalk_zones_np, (0, 0, 255))
    cv2.addWeighted(overlay, 0.2, im0, 0.8, 0, im0)
    cv2.line(im0, (road_divider_x, 0), (road_divider_x, h), (0, 255, 255), 1)  # show divider reference
    frame_track_speeds = []
    
    # --- DIRECTIONAL SIDEWALK ZONE CHECK ---
    if counter.track_ids:
        confs = counter.confs if getattr(counter, "confs", None) is not None else [None] * len(counter.track_ids)
        for track_id, box, cls, conf in zip(counter.track_ids, counter.boxes, counter.clss, confs):
            
            # Explicitly cast to float to ensure cv2 compatibility
            x_center = float((box[0] + box[2]) / 2)
            y_center = float((box[1] + box[3]) / 2)
            x1 = int(float(box[0]))
            y1 = int(float(box[1]))

            if track_id not in tracklet_kf:
                tracklet_kf[track_id] = init_kalman(x_center, y_center)
            kf = tracklet_kf[track_id]
            kf.predict()
            kf_state = kf.correct(np.array([[x_center], [y_center]], dtype=np.float32))
            kf_x = float(kf_state[0, 0])
            kf_y = float(kf_state[1, 0])
            current_point = (kf_x, kf_y)

            # --- Tracklet feature updates ---
            cls_mapped = remap_vehicle_class(cls, box, w, h, conf)
            tracklet_frames_seen[track_id] += 1
            tracklet_class_votes[track_id][int(cls_mapped)] += 1
            if conf is not None:
                tracklet_conf_sum[track_id] += float(conf)

            if track_id not in tracklet_first_last:
                tracklet_first_last[track_id] = [frame_count, frame_count]
            else:
                tracklet_first_last[track_id][1] = frame_count

            history = tracklet_history[track_id]
            step_distance = 0.0
            had_prev = bool(history)
            if had_prev:
                prev_hx, prev_hy = history[-1]
                step_distance = float(np.hypot(kf_x - prev_hx, kf_y - prev_hy))
                tracklet_distance[track_id] += step_distance
            history.append((kf_x, kf_y))

            # Approximate speed in px/s from inter-frame displacement
            instant_speed = step_distance * fps
            prev_speed = tracklet_prev_speed.get(track_id)
            accel = ((instant_speed - prev_speed) * fps) if prev_speed is not None else 0.0
            tracklet_prev_speed[track_id] = instant_speed
            frame_track_speeds.append(instant_speed)
            if had_prev:
                tracklet_step_distance_values[track_id].append(step_distance)
                tracklet_speed_values[track_id].append(instant_speed)
                if prev_speed is not None:
                    tracklet_accel_values[track_id].append(float(accel))
            
            # Check if point is inside each zone and on the correct half of the road
            is_in_left  = (
                cv2.pointPolygonTest(sidewalk_zones_np[0], current_point, False) >= 0
                and kf_x <= road_divider_x - divider_buffer_px
            )
            is_in_right = (
                cv2.pointPolygonTest(sidewalk_zones_np[1], current_point, False) >= 0
                and kf_x >= road_divider_x + divider_buffer_px
            )

            # Determine movement direction using previous position
            is_moving_down = False  # y increasing -> moving down in image
            is_moving_up   = False  # y decreasing -> moving up in image

            if track_id in prev_positions:
                prev_x, prev_y = prev_positions[track_id]
                dy = kf_y - prev_y  # positive = moving down, negative = moving up

                if dy > MOVEMENT_THRESHOLD:
                    is_moving_down = True
                elif dy < -MOVEMENT_THRESHOLD:
                    is_moving_up = True
                # else: |dy| <= threshold -> considered stopped, no violation
            
            # Update position history
            prev_positions[track_id] = (kf_x, kf_y)

            # Flag violation only if in zone AND moving the wrong direction
            is_left_violation  = is_in_left  and is_moving_down
            is_right_violation = is_in_right and is_moving_up

            if track_id not in violation_ids:
                if is_left_violation:
                    left_sidewalk_violations += 1
                    violation_ids.add(track_id)
                elif is_right_violation:
                    right_sidewalk_violations += 1
                    violation_ids.add(track_id)

            # Highlight active violators (currently in zone + wrong direction)
            if is_left_violation or is_right_violation:
                cv2.circle(im0, (int(kf_x), int(kf_y)), 12, (0, 0, 255), -1)
                cv2.putText(im0, "VIOLATION", (x1, max(15, y1 - 10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
            # Dim highlight for previously recorded violators still in frame
            elif track_id in violation_ids and (is_in_left or is_in_right):
                cv2.circle(im0, (int(kf_x), int(kf_y)), 8, (0, 165, 255), 2)

            # Always show confidence near each tracked box
            dominant_cls = max(tracklet_class_votes[track_id], key=tracklet_class_votes[track_id].get)
            dominant_name = class_name_map.get(int(dominant_cls), f"cls{int(dominant_cls)}")
            avg_conf = (tracklet_conf_sum[track_id] / tracklet_frames_seen[track_id]) if tracklet_frames_seen[track_id] else 0.0

            if len(history) >= 2:
                pts = np.array([(int(px), int(py)) for px, py in history], dtype=np.int32)
                cv2.polylines(im0, [pts], False, (255, 200, 0), 2)

            cv2.putText(
                im0,
                f"ID {int(track_id)} {dominant_name} AvgC:{avg_conf:.2f}",
                (x1, max(20, y1 - 30)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (255, 255, 0),
                2,
            )
            cv2.putText(
                im0,
                f"v:{instant_speed:.1f}px/s a:{accel:.1f}px/s2",
                (x1, max(40, y1 - 10)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0, 220, 255),
                2,
            )

    # Keep track state alive for short misses to reduce blinking
    active_ids = set(counter.track_ids) if counter.track_ids else set()
    for tid in active_ids:
        tracklet_missed_frames[tid] = 0

    for tid in list(tracklet_kf.keys()):
        if tid in active_ids:
            continue

        tracklet_missed_frames[tid] += 1
        if tracklet_missed_frames[tid] <= MAX_MISSED_FRAMES:
            continue

        del tracklet_kf[tid]
        if tid in prev_positions:
            del prev_positions[tid]
        if tid in tracklet_history:
            del tracklet_history[tid]
        if tid in tracklet_prev_speed:
            del tracklet_prev_speed[tid]
        if tid in tracklet_missed_frames:
            del tracklet_missed_frames[tid]

    # Compute real-time FPS (smoothed for readability)
    frame_time = max(time.time() - frame_start_time, 1e-6)
    current_fps = 1.0 / frame_time
    fps_smooth = current_fps if fps_smooth is None else (0.9 * fps_smooth + 0.1 * current_fps)

    # Add custom dashboard text
    total_vehicle_count = counter.in_count + counter.out_count
    active_tracks = len(counter.track_ids) if counter.track_ids else 0
    avg_track_speed = (sum(frame_track_speeds) / len(frame_track_speeds)) if active_tracks else 0.0
    cv2.putText(im0, f"Processing FPS: {fps_smooth:.2f}", 
                (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    cv2.putText(im0, f"Video FPS (source): {fps:.2f}", 
                (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (180, 180, 180), 2)
    cv2.putText(im0, f"Total Vehicles: {total_vehicle_count}", 
                (20, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    cv2.putText(im0, f"Active Tracks: {active_tracks} | Avg speed: {avg_track_speed:.1f}px/s", 
                (20, 180), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 220, 120), 2)
    cv2.putText(im0, f"Left Violations (down): {left_sidewalk_violations}", 
                (20, 140), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
    cv2.putText(im0, f"Right Violations (up): {right_sidewalk_violations}", 
                (w - 520, 140), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
    
    # Write to video
    video_writer.write(im0)

    # Progress update
    if frame_count % 30 == 0:
        print(f"Processed {frame_count} frames... | FPS: {fps_smooth:.2f}", end='\r')

    # # Safety break for testing (remove to run full video)
    # if frame_count > 30: 
    #     print("\nReached frame limit.")
    #     break

# --- 5. CLEANUP ---
cap.release()
video_writer.release()
cv2.destroyAllWindows()

print(f"\nProcessing finished. Saved to '{output_filename}'")
print(f"Total Vehicles Counted: {counter.in_count + counter.out_count}")
print(f"Left Sidewalk Violations (moving down): {left_sidewalk_violations}")
print(f"Right Sidewalk Violations (moving up):  {right_sidewalk_violations}")
print(f"Total Unique Violators: {len(violation_ids)}")

# --- 6. FEATURE EXTRACTION (ID + speed/distance avg, min, max) ---
feature_rows = []
for tid in sorted(tracklet_frames_seen.keys()):
    speed_vals = tracklet_speed_values.get(tid, [])
    accel_vals = tracklet_accel_values.get(tid, [])
    dist_vals = tracklet_step_distance_values.get(tid, [])
    cls_votes = tracklet_class_votes.get(tid, {})
    dominant_cls = max(cls_votes, key=cls_votes.get) if cls_votes else -1

    speed_avg = float(np.mean(speed_vals)) if speed_vals else 0.0
    speed_min = float(np.min(speed_vals)) if speed_vals else 0.0
    speed_max = float(np.max(speed_vals)) if speed_vals else 0.0
    dist_avg = float(np.mean(dist_vals)) if dist_vals else 0.0
    dist_min = float(np.min(dist_vals)) if dist_vals else 0.0
    dist_max = float(np.max(dist_vals)) if dist_vals else 0.0
    accel_avg = float(np.mean(accel_vals)) if accel_vals else 0.0
    accel_min = float(np.min(accel_vals)) if accel_vals else 0.0
    accel_max = float(np.max(accel_vals)) if accel_vals else 0.0

    feature_rows.append({
        "id": int(tid),
        "dominant_class_id": int(dominant_cls),
        "speed_avg_px_s": speed_avg,
        "speed_min_px_s": speed_min,
        "speed_max_px_s": speed_max,
        "distance_avg_px": dist_avg,
        "distance_min_px": dist_min,
        "distance_max_px": dist_max,
        "accel_avg_px_s2": accel_avg,
        "accel_min_px_s2": accel_min,
        "accel_max_px_s2": accel_max,
    })

tracklet_feature_df = pd.DataFrame(feature_rows)
if not tracklet_feature_df.empty:
    tracklet_feature_df = tracklet_feature_df.sort_values("id").reset_index(drop=True)
    print("Tracklet feature extraction complete:")
    display(tracklet_feature_df)
    tracklet_feature_df.to_csv("tracklet_features.csv", index=False)
    print("Saved features to tracklet_features.csv")
else:
    print("No tracklet features were extracted.")

if tracklet_distance:
    print("Top tracklets by distance (pixels):")
    top_tracklets = sorted(tracklet_distance.items(), key=lambda x: x[1], reverse=True)[:5]
    for tid, dist in top_tracklets:
        first_f, last_f = tracklet_first_last.get(tid, [frame_count, frame_count])
        duration_s = (last_f - first_f + 1) / max(fps, 1)
        avg_conf = (tracklet_conf_sum[tid] / tracklet_frames_seen[tid]) if tracklet_frames_seen[tid] else 0.0
        cls_votes = tracklet_class_votes[tid]
        dominant_cls = max(cls_votes, key=cls_votes.get) if cls_votes else -1
        cls_name = class_name_map.get(int(dominant_cls), f"cls{int(dominant_cls)}")
        print(f"  ID {int(tid)} | dist={dist:.1f}px | dur={duration_s:.2f}s | avg_conf={avg_conf:.2f} | class={cls_name}({int(dominant_cls)})")

CUDA status: True
GPU Name: NVIDIA GeForce RTX 4050 Laptop GPU
Current VRAM Allocated: 331.86 MB
Ultralytics Solutions:  {'source': None, 'model': 'rtdetr-l.pt', 'classes': [2, 3, 5, 7], 'show_conf': True, 'show_labels': True, 'region': [(0, 696), (1920, 696)], 'colormap': 21, 'show_in': True, 'show_out': True, 'up_angle': 145.0, 'down_angle': 90, 'kpts': [6, 8, 10], 'analytics_type': 'line', 'figsize': (12.8, 7.2), 'blur_ratio': 0.5, 'vision_point': (20, 20), 'crop_dir': 'cropped-detections', 'json_file': None, 'line_width': 2, 'records': 5, 'fps': 30.0, 'max_hist': 5, 'meter_per_pixel': 0.05, 'max_speed': 120, 'show': False, 'iou': 0.7, 'conf': 0.25, 'device': None, 'max_det': 300, 'half': False, 'tracker': 'rtdetr_tuned.yaml', 'verbose': True, 'data': 'images'}
Starting combined monitoring on CCTV_Sleman.mp4...
Left zone violation = moving DOWN | Right zone violation = moving UP
Tracker config: rtdetr_tuned.yaml | missed frame grace: 45
0: 1080x1920 2.5ms, 7 car, 1 truck, 3 motorcyc

,id,dominant_class_id,speed_avg_px_s,speed_min_px_s,speed_max_px_s,distance_avg_px,distance_min_px,distance_max_px,accel_avg_px_s2,accel_min_px_s2,accel_max_px_s2
0,1,2,30.397940,0.050944,209.062969,1.013265,0.001698,6.968766,0.200593,-1648.189391,4119.006199
1,2,2,111.654502,2.872126,689.455743,3.721817,0.095738,22.981858,5.784074,-6356.800351,16042.641202
2,3,2,390.558158,8.264454,972.299820,13.018605,0.275482,32.409994,296.793751,-8465.177075,17287.461826
3,4,7,2.084602,0.011138,102.955677,0.069487,0.000371,3.431856,0.009460,-1012.589913,2682.811018
4,5,2,191.142321,1.794966,1600.092280,6.371411,0.059832,53.336409,85.268653,-15900.780171,21447.164995
...,...,...,...,...,...,...,...,...,...,...,...
366,631,3,492.777626,286.756889,706.090950,16.425921,9.558563,23.536365,870.821399,-7593.271655,10701.103951
367,632,3,374.063666,196.975546,645.655428,12.468789,6.565852,21.521848,520.279022,-6712.156237,6566.169862
368,633,3,117.998018,26.169365,187.383825,3.933267,0.872312,6.246127,122.557529,-1912.472767,2713.373048
369,635,3,465.251943,283.560839,730.265030,15.508398,9.452028,24.342168,405.942827,-11071.563591,13623.458007


Saved features to tracklet_features.csv
Top tracklets by distance (pixels):
  ID 6 | dist=1830.6px | dur=129.97s | avg_conf=0.90 | class=car(2)
  ID 563 | dist=1268.4px | dur=10.03s | avg_conf=0.74 | class=car(2)
  ID 422 | dist=1161.2px | dur=11.90s | avg_conf=0.79 | class=car(2)
  ID 157 | dist=1152.1px | dur=10.47s | avg_conf=0.72 | class=car(2)
  ID 209 | dist=1147.0px | dur=6.27s | avg_conf=0.74 | class=car(2)


[Full YOLO model testing video](https://youtu.be/gKysf4sNiO8)
![yolo sample](./sample.png)
![violator sample](./Violator.png)

## Temporal VAE Anomaly Section
This section is separate from the RT-DETR detection/tracking block above.
It trains and runs two Temporal VAEs split by road divider:
- Left-side model focuses only on left roadway region.
- Right-side model focuses only on right roadway region.

Execution order:
1. Run Cell 5a.5 (build divider ROIs and shared defaults).
2. Run Cell 5b (generate synthetic wrong-way samples).
3. Run Cell 5c (train the left/right VAEs).
4. Run Cell 5d (calibrate left/right thresholds).
5. Run Cell 7 (fuse tracker + VAE + wrong-way rules + export alert log).

Notes:
- Training source folders are separated: `vae_split_samples/left` and `vae_split_samples/right`.
- Divider source uses `road_divider_x` from the RT-DETR section when available.
- If `road_divider_x` is missing, divider is estimated from frame width (`w / 1.88`).
- Separate weights are used for each side (`temporal_vae_left_weights.pt`, `temporal_vae_right_weights.pt`).

Main outputs:
- `tracklet_features.csv`
- `vae_anomaly_scores_split_lr.csv`
- `fused_alert_log.csv`
- `vae_bytetrack_anomaly_overlay.mp4`

In [1]:
from pathlib import Path
import cv2
import torch
from vae_anomaly_module import prepare_vae_context
from ultralytics import YOLO
import warnings
import numpy as np
import pandas as pd
from collections import defaultdict

# --- HELPER DEFINITIONS ---
VIDEO_SUFFIXES = {".mp4", ".avi", ".mov", ".mkv", ".m4v"}

# --- 1) DATA SOURCES ---
VAE_NORMAL_SOURCE_LEFT = "vae_split_samples/left"
VAE_NORMAL_SOURCE_RIGHT = "vae_split_samples/right"
VAE_TEST_SOURCE = "CCTV_Sleman.mp4"  # set None to split eval from each side source
VAE_WEIGHTS_LEFT_PATH = "temporal_vae_left_weights.pt"
VAE_WEIGHTS_RIGHT_PATH = "temporal_vae_right_weights.pt"

# --- 2) MODEL / DATA SETTINGS ---
VAE_IMG_SIZE = 64
VAE_SEQ_LEN = 16
VAE_FRAME_STEP = 2
VAE_SEQ_STRIDE = 2
VAE_MAX_FRAMES_PER_VIDEO = 2500
VAE_MAX_SEQUENCES = 1800
VAE_BATCH_SIZE = 8
VAE_LATENT_DIM = 64
VAE_HIDDEN_DIM = 128
VAE_EMB_DIM = 128
# --- 1b) TRACKLET EXTRACTION (for hybrid VAE) ---
# When True, extract aggregated tracklet features from the normal-source videos and
# save per-side CSVs that can be merged into VAE sequence records (hybrid features).
VAE_EXTRACT_TRACKLETS = True
VAE_TRACKLET_OUT_LEFT = "vae_tracklet_features_left.csv"
VAE_TRACKLET_OUT_RIGHT = "vae_tracklet_features_right.csv"
VAE_TRACKLET_DETECTOR = str(globals().get("TRACK_MODEL_PATH", "yolo11m.pt"))
VAE_TRACKLET_TRACKER_CFG = str(globals().get("TRACKER_CFG_PATH", "bytetrack_tuned.yaml"))

# --- 3) TRAIN-IF-MISSING SETTINGS ---
VAE_TRAIN_IF_MISSING = True
VAE_TRAIN_EPOCHS = 5
VAE_TRAIN_LR = 1e-3
VAE_TRAIN_KL_WEIGHT = 0.05

# --- 4) LEFT/RIGHT SPLIT SETTINGS ---
VAE_DIVIDER_BUFFER = 0

VIDEO_SUFFIXES = {".mp4", ".avi", ".mov", ".mkv", ".m4v"}

def has_video_content(path: Path) -> bool:
    if path.is_file():
        return path.suffix.lower() in VIDEO_SUFFIXES
    if path.is_dir():
        return any(p.is_file() and p.suffix.lower() in VIDEO_SUFFIXES for p in path.iterdir())
    return False

def pick_first_video(path: Path) -> Path | None:
    if path.is_file() and path.suffix.lower() in VIDEO_SUFFIXES:
        return path
    if path.is_dir():
        for p in sorted(path.iterdir()):
            if p.is_file() and p.suffix.lower() in VIDEO_SUFFIXES:
                return p
    return None

left_source_path = Path(VAE_NORMAL_SOURCE_LEFT)
right_source_path = Path(VAE_NORMAL_SOURCE_RIGHT)

if not has_video_content(left_source_path):
    raise FileNotFoundError(
        f"Left normal source has no videos: {left_source_path}. "
        "Create clips under 'vae_split_samples/left' first."
    )

if not has_video_content(right_source_path):
    raise FileNotFoundError(
        f"Right normal source has no videos: {right_source_path}. "
        "Create clips under 'vae_split_samples/right' first."
    )

if VAE_TEST_SOURCE is not None and not Path(VAE_TEST_SOURCE).exists():
    print(f"Test source '{VAE_TEST_SOURCE}' not found. Using split-from-normal mode instead.")
    VAE_TEST_SOURCE = None

def resolve_timeline_source() -> Path:
    if VAE_TEST_SOURCE is not None:
        test_path = Path(VAE_TEST_SOURCE)
        if test_path.is_file():
            return test_path

    for src in [left_source_path, right_source_path]:
        first_video = pick_first_video(src)
        if first_video is not None:
            return first_video

    raise FileNotFoundError(
        "Could not resolve timeline source from test video or split normal folders."
    )

timeline_path = resolve_timeline_source()
vae_timeline_source = str(timeline_path)

cap_meta = cv2.VideoCapture(vae_timeline_source)
if not cap_meta.isOpened():
    raise RuntimeError(f"Cannot open timeline source: {vae_timeline_source}")
frame_w = int(cap_meta.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_h = int(cap_meta.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap_meta.release()

divider_guess = int(frame_w / 1.88)
divider_x = int(globals().get("road_divider_x", divider_guess))
divider_x = max(1, min(frame_w - 1, divider_x))
buffer_px = max(0, int(VAE_DIVIDER_BUFFER))

left_x2 = max(1, min(frame_w - 1, divider_x - buffer_px))
right_x1 = min(frame_w - 1, max(0, divider_x + buffer_px))

if left_x2 < 16 or (frame_w - right_x1) < 16:
    raise RuntimeError(
        f"Invalid split widths with divider_x={divider_x}, buffer={buffer_px}, frame_w={frame_w}."
    )

VAE_LEFT_ROI = (0, 0, left_x2, frame_h)
VAE_RIGHT_ROI = (right_x1, 0, frame_w, frame_h)

print(f"Timeline source: {vae_timeline_source}")
print(f"Frame size: {frame_w}x{frame_h}")
print(f"Divider x: {divider_x} | Buffer: {buffer_px}")
print(f"Left ROI:  {VAE_LEFT_ROI}")
print(f"Right ROI: {VAE_RIGHT_ROI}")

vae_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# --- Tracklet extraction helper (lightweight aggregation) ---
def _extract_tracklet_features_from_folder(src_folder: str, out_csv: str,
                                        detector_ref: str, tracker_cfg_ref: str,
                                        classes=None, conf=0.30, iou=0.50,
                                        max_videos=None):
    src = Path(src_folder)
    out = Path(out_csv)
    if out.exists():
        print(f'Existing tracklet CSV {out} found, skipping extraction.')
        return
    if not src.exists():
        raise FileNotFoundError(f'source folder missing: {src}')
    model = YOLO(detector_ref)
    rows = []
    video_files = [p for p in sorted(src.iterdir()) if p.is_file() and p.suffix.lower() in VIDEO_SUFFIXES]
    if max_videos is not None:
        video_files = video_files[:max_videos]
    for vid in video_files:
        print(f'Extracting tracklets from: {vid.name}')
        try:
            stream = model.track(source=str(vid), stream=True, persist=True,
                                 tracker=str(tracker_cfg_ref), classes=classes, conf=conf, iou=iou, verbose=False)
        except Exception as e:
            warnings.warn(f'Failed to start tracker on {vid}: {e}')
            continue
        # per-video accumulators
        frames_seen = defaultdict(int)
        conf_sum = defaultdict(float)
        first_last = {}
        class_votes = defaultdict(lambda: defaultdict(int))
        step_dist_acc = defaultdict(float)
        prev_pos = {}
        for res in stream:
            boxes = getattr(res, 'boxes', None)
            if boxes is None or len(boxes) == 0:
                continue
            xyxy = boxes.xyxy.detach().cpu().numpy() if boxes.xyxy is not None else np.zeros((0, 4), dtype=np.float32)
            ids = boxes.id.int().detach().cpu().numpy() if boxes.id is not None else np.full((len(xyxy),), -1, dtype=np.int32)
            confs = boxes.conf.detach().cpu().numpy() if boxes.conf is not None else np.zeros((len(xyxy),), dtype=np.float32)
            clss = boxes.cls.int().detach().cpu().numpy() if boxes.cls is not None else np.full((len(xyxy),), -1, dtype=np.int32)
            for (x1, y1, x2, y2), tid, confv, clsv in zip(xyxy, ids, confs, clss):
                tid = int(tid)
                cx = float(0.5 * (x1 + x2))
                cy = float(0.5 * (y1 + y2))
                frames_seen[tid] += 1
                conf_sum[tid] += float(confv) if confv is not None else 0.0
                class_votes[tid][int(clsv)] += 1
                if tid not in first_last:
                    first_last[tid] = [int(getattr(res, 'frame_id', -1)), int(getattr(res, 'frame_id', -1))]
                else:
                    first_last[tid][1] = int(getattr(res, 'frame_id', -1))
                if tid in prev_pos:
                    px, py = prev_pos[tid]
                    step = float(np.hypot(cx - px, cy - py))
                    step_dist_acc[tid] += step
                prev_pos[tid] = (cx, cy)
        # summarize per-track
        for tid, cnt in frames_seen.items():
            cls_votes = class_votes.get(tid, {})
            dom_cls = max(cls_votes, key=cls_votes.get) if cls_votes else -1
            avg_conf = float(conf_sum.get(tid, 0.0) / max(1, cnt))
            dist = float(step_dist_acc.get(tid, 0.0))
            ff, lf = first_last.get(tid, [-1, -1])
            dur = max(0, lf - ff + 1) if ff >= 0 else 0
            rows.append({
                'video': vid.name,
                'id': int(tid),
                'dominant_class_id': int(dom_cls),
                'frames_seen': int(cnt),
                'avg_conf': avg_conf,
                'distance_px': dist,
                'first_frame': int(ff),
                'last_frame': int(lf),
                'duration_frames': int(dur),
            })
        try:
            # ensure stream closed
            stream.close()
        except Exception:
            pass
    if rows:
        df = pd.DataFrame(rows)
        # Handle all-zero columns: drop non-informative all-zero columns, and for id/class set to -1 if all zeros.
        for col in list(df.columns):
            if col in ('id', 'dominant_class_id'):
                if (df[col] == 0).all():
                    df[col] = df[col].apply(lambda v: -1)
            else:
                if pd.api.types.is_numeric_dtype(df[col]) and (df[col] == 0).all():
                    df.drop(columns=[col], inplace=True)
        df.to_csv(out, index=False)
        print(f'Saved tracklet features to: {out}')
    else:
        print(f'No tracklets extracted for source: {src}')

# Optionally extract per-side tracklet features from normal sources (this can be time-consuming).
if VAE_EXTRACT_TRACKLETS:
    try:
        _extract_tracklet_features_from_folder(VAE_NORMAL_SOURCE_LEFT, VAE_TRACKLET_OUT_LEFT,
                                                detector_ref=VAE_TRACKLET_DETECTOR,
                                                tracker_cfg_ref=VAE_TRACKLET_TRACKER_CFG,
                                                classes=globals().get('target_classes', None))
    except Exception as e:
        warnings.warn(f'Left tracklet extraction failed: {e}')
    try:
        _extract_tracklet_features_from_folder(VAE_NORMAL_SOURCE_RIGHT, VAE_TRACKLET_OUT_RIGHT,
                                                detector_ref=VAE_TRACKLET_DETECTOR,
                                                tracker_cfg_ref=VAE_TRACKLET_TRACKER_CFG,
                                                classes=globals().get('target_classes', None))
    except Exception as e:
        warnings.warn(f'Right tracklet extraction failed: {e}')
# Build left-side specialized model/loaders (trained from left folder)
vae_model_left, _, normal_loader_left, test_loader_left = prepare_vae_context(
    normal_source=VAE_NORMAL_SOURCE_LEFT,
    test_source=VAE_TEST_SOURCE,
    weights_path=VAE_WEIGHTS_LEFT_PATH,
    img_size=VAE_IMG_SIZE,
    seq_len=VAE_SEQ_LEN,
    frame_step=VAE_FRAME_STEP,
    seq_stride=VAE_SEQ_STRIDE,
    max_frames_per_video=VAE_MAX_FRAMES_PER_VIDEO,
    max_sequences=VAE_MAX_SEQUENCES,
    batch_size=VAE_BATCH_SIZE,
    latent_dim=VAE_LATENT_DIM,
    hidden_dim=VAE_HIDDEN_DIM,
    emb_dim=VAE_EMB_DIM,
    train_if_missing=VAE_TRAIN_IF_MISSING,
    train_epochs=VAE_TRAIN_EPOCHS,
    train_lr=VAE_TRAIN_LR,
    train_kl_weight=VAE_TRAIN_KL_WEIGHT,
    crop_roi=VAE_LEFT_ROI,
    device=vae_device,
)

# Build right-side specialized model/loaders (trained from right folder)
vae_model_right, _, normal_loader_right, test_loader_right = prepare_vae_context(
    normal_source=VAE_NORMAL_SOURCE_RIGHT,
    test_source=VAE_TEST_SOURCE,
    weights_path=VAE_WEIGHTS_RIGHT_PATH,
    img_size=VAE_IMG_SIZE,
    seq_len=VAE_SEQ_LEN,
    frame_step=VAE_FRAME_STEP,
    seq_stride=VAE_SEQ_STRIDE,
    max_frames_per_video=VAE_MAX_FRAMES_PER_VIDEO,
    max_sequences=VAE_MAX_SEQUENCES,
    batch_size=VAE_BATCH_SIZE,
    latent_dim=VAE_LATENT_DIM,
    hidden_dim=VAE_HIDDEN_DIM,
    emb_dim=VAE_EMB_DIM,
    train_if_missing=VAE_TRAIN_IF_MISSING,
    train_epochs=VAE_TRAIN_EPOCHS,
    train_lr=VAE_TRAIN_LR,
    train_kl_weight=VAE_TRAIN_KL_WEIGHT,
    crop_roi=VAE_RIGHT_ROI,
    device=vae_device,
)

print("Dual VAE setup complete.")
print(f"Device: {vae_device}")
print(f"Left normal source:  {VAE_NORMAL_SOURCE_LEFT}")
print(f"Right normal source: {VAE_NORMAL_SOURCE_RIGHT}")
print(f"Test source: {VAE_TEST_SOURCE if VAE_TEST_SOURCE is not None else '[split from each normal folder]'}")
print(f"Left weights:  {VAE_WEIGHTS_LEFT_PATH}")
print(f"Right weights: {VAE_WEIGHTS_RIGHT_PATH}")
print(f"Left  loaders  -> normal:{len(normal_loader_left)} test:{len(test_loader_left)}")
print(f"Right loaders  -> normal:{len(normal_loader_right)} test:{len(test_loader_right)}")

Timeline source: CCTV_Sleman.mp4
Frame size: 1920x1080
Divider x: 1021 | Buffer: 0
Left ROI:  (0, 0, 1021, 1080)
Right ROI: (1021, 0, 1920, 1080)
Existing tracklet CSV vae_tracklet_features_left.csv found, skipping extraction.
Existing tracklet CSV vae_tracklet_features_right.csv found, skipping extraction.
Dual VAE setup complete.
Device: cuda
Left normal source:  vae_split_samples/left
Right normal source: vae_split_samples/right
Test source: CCTV_Sleman.mp4
Left weights:  temporal_vae_left_weights.pt
Right weights: temporal_vae_right_weights.pt
Left  loaders  -> normal:225 test:121
Right loaders  -> normal:225 test:121


In [2]:
# Cell 5a.5: Bootstrap defaults for downstream cells
from pathlib import Path
import cv2
import torch
import numpy as np
import pandas as pd
import warnings

VIDEO_SUFFIXES = globals().get("VIDEO_SUFFIXES", {".mp4", ".avi", ".mov", ".mkv", ".m4v"})
VAE_NORMAL_SOURCE_LEFT = str(globals().get("VAE_NORMAL_SOURCE_LEFT", "vae_split_samples/left"))
VAE_NORMAL_SOURCE_RIGHT = str(globals().get("VAE_NORMAL_SOURCE_RIGHT", "vae_split_samples/right"))
VAE_TEST_SOURCE = globals().get("VAE_TEST_SOURCE", "CCTV_Sleman.mp4")
VAE_IMG_SIZE = int(globals().get("VAE_IMG_SIZE", 64))
VAE_SEQ_LEN = int(globals().get("VAE_SEQ_LEN", 16))
VAE_FRAME_STEP = int(globals().get("VAE_FRAME_STEP", 2))
VAE_SEQ_STRIDE = int(globals().get("VAE_SEQ_STRIDE", 2))
VAE_MAX_FRAMES_PER_VIDEO = int(globals().get("VAE_MAX_FRAMES_PER_VIDEO", 2500))
VAE_MAX_SEQUENCES = int(globals().get("VAE_MAX_SEQUENCES", 1800))
VAE_BATCH_SIZE = int(globals().get("VAE_BATCH_SIZE", 8))
VAE_LATENT_DIM = int(globals().get("VAE_LATENT_DIM", 64))
VAE_HIDDEN_DIM = int(globals().get("VAE_HIDDEN_DIM", 128))
VAE_EMB_DIM = int(globals().get("VAE_EMB_DIM", 128))
VAE_TRAIN_KL_WEIGHT = float(globals().get("VAE_TRAIN_KL_WEIGHT", 0.05))
KL_WEIGHT = float(globals().get("KL_WEIGHT", VAE_TRAIN_KL_WEIGHT))

if "vae_device" not in globals():
    vae_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if "vae_timeline_source" not in globals():
    timeline_candidate = Path(VAE_TEST_SOURCE) if VAE_TEST_SOURCE else None
    if timeline_candidate is None or not timeline_candidate.exists():
        for candidate in (Path(VAE_NORMAL_SOURCE_LEFT), Path(VAE_NORMAL_SOURCE_RIGHT)):
            if candidate.is_file() and candidate.suffix.lower() in VIDEO_SUFFIXES:
                timeline_candidate = candidate
                break
            if candidate.is_dir():
                videos = sorted([p for p in candidate.iterdir() if p.is_file() and p.suffix.lower() in VIDEO_SUFFIXES])
                if videos:
                    timeline_candidate = videos[0]
                    break
    if timeline_candidate is not None and timeline_candidate.exists():
        vae_timeline_source = str(timeline_candidate)

if "vae_timeline_source" in globals() and ("VAE_LEFT_ROI" not in globals() or "VAE_RIGHT_ROI" not in globals()):
    cap_boot = cv2.VideoCapture(str(vae_timeline_source))
    if cap_boot.isOpened():
        fw = int(cap_boot.get(cv2.CAP_PROP_FRAME_WIDTH))
        fh = int(cap_boot.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap_boot.release()
        divider_x_boot = int(globals().get("road_divider_x", int(fw / 1.88)))
        divider_x_boot = max(1, min(fw - 1, divider_x_boot))
        buffer_boot = int(globals().get("VAE_DIVIDER_BUFFER", 0))
        left_x2_boot = max(1, min(fw - 1, divider_x_boot - buffer_boot))
        right_x1_boot = min(fw - 1, max(0, divider_x_boot + buffer_boot))
        if "VAE_LEFT_ROI" not in globals():
            VAE_LEFT_ROI = (0, 0, left_x2_boot, fh)
        if "VAE_RIGHT_ROI" not in globals():
            VAE_RIGHT_ROI = (right_x1_boot, 0, fw, fh)

threshold_left_vae = globals().get("threshold_left_vae", globals().get("THR_LEFT_VAE", None))
threshold_right_vae = globals().get("threshold_right_vae", globals().get("THR_RIGHT_VAE", None))
threshold_left_gan = globals().get("threshold_left_gan", globals().get("THR_LEFT_GAN", None))
threshold_right_gan = globals().get("threshold_right_gan", globals().get("THR_RIGHT_GAN", None))

print("Bootstrap defaults ready for downstream cells.")

Bootstrap defaults ready for downstream cells.


In [3]:
# Cell 5b.5: Shared imports and prerequisites for downstream analysis cells
import numpy as np
import torch

required_after_5b = [
    "vae_device", "VAE_LEFT_ROI", "VAE_RIGHT_ROI",
    "VAE_SEQ_LEN", "VAE_SEQ_STRIDE", "VAE_IMG_SIZE",
    "VAE_TRAIN_KL_WEIGHT",
    "vae_model_left", "vae_model_right",
    "normal_loader_left", "normal_loader_right",
    "test_loader_left", "test_loader_right",
]
missing_after_5b = [name for name in required_after_5b if name not in globals()]
if missing_after_5b:
    print("Missing prerequisites for later cells:")
    for name in missing_after_5b:
        print(f"  - {name}")
    print("Run Cell 5 first (dual VAE setup) before running GAN/diagnostic cells.")
else:
    print("Downstream prerequisites are available.")

Downstream prerequisites are available.


In [4]:
# Cell 5c: Train GANomaly Models on merged normal + synthetic wrong-way data
import importlib
import sys
if 'vae_anomaly_module' in sys.modules:
    importlib.reload(sys.modules['vae_anomaly_module'])

from vae_anomaly_module import collect_sequences, make_loader

print("Collecting sequences...")

# Collect normal + synthetic wrong-way sequences (merged)
normal_left_seqs = collect_sequences(
    VAE_NORMAL_SOURCE_LEFT,
    img_size=VAE_IMG_SIZE, seq_len=VAE_SEQ_LEN, frame_step=VAE_FRAME_STEP, seq_stride=VAE_SEQ_STRIDE,
    max_frames_per_video=VAE_MAX_FRAMES_PER_VIDEO, max_sequences=VAE_MAX_SEQUENCES,
    crop_roi=VAE_LEFT_ROI
)
synthetic_left_seqs = collect_sequences(
    "vae_wrong_way_samples/left",
    img_size=VAE_IMG_SIZE, seq_len=VAE_SEQ_LEN, frame_step=VAE_FRAME_STEP, seq_stride=VAE_SEQ_STRIDE,
    max_frames_per_video=VAE_MAX_FRAMES_PER_VIDEO, max_sequences=VAE_MAX_SEQUENCES,
    crop_roi=VAE_LEFT_ROI
)
train_left_merged = np.concatenate([normal_left_seqs, synthetic_left_seqs], axis=0)
train_left_loader_merged = make_loader(train_left_merged, batch_size=VAE_BATCH_SIZE, shuffle=True)

# Same for right
normal_right_seqs = collect_sequences(
    VAE_NORMAL_SOURCE_RIGHT,
    img_size=VAE_IMG_SIZE, seq_len=VAE_SEQ_LEN, frame_step=VAE_FRAME_STEP, seq_stride=VAE_SEQ_STRIDE,
    max_frames_per_video=VAE_MAX_FRAMES_PER_VIDEO, max_sequences=VAE_MAX_SEQUENCES,
    crop_roi=VAE_RIGHT_ROI
)
synthetic_right_seqs = collect_sequences(
    "vae_wrong_way_samples/right",
    img_size=VAE_IMG_SIZE, seq_len=VAE_SEQ_LEN, frame_step=VAE_FRAME_STEP, seq_stride=VAE_SEQ_STRIDE,
    max_frames_per_video=VAE_MAX_FRAMES_PER_VIDEO, max_sequences=VAE_MAX_SEQUENCES,
    crop_roi=VAE_RIGHT_ROI
)
train_right_merged = np.concatenate([normal_right_seqs, synthetic_right_seqs], axis=0)
train_right_loader_merged = make_loader(train_right_merged, batch_size=VAE_BATCH_SIZE, shuffle=True)

print(f"Left: normal={len(normal_left_seqs)} + synthetic={len(synthetic_left_seqs)} = {len(train_left_merged)} total")
print(f"Right: normal={len(normal_right_seqs)} + synthetic={len(synthetic_right_seqs)} = {len(train_right_merged)} total")

# Initialize & train GANomaly models
print("Initializing GANomaly models...")
# Reload again right before importing GANomaly to ensure fresh import
if 'vae_anomaly_module' in sys.modules:
    importlib.reload(sys.modules['vae_anomaly_module'])
from vae_anomaly_module import GANomaly, train_ganomaly

gan_model_left = GANomaly(seq_len=VAE_SEQ_LEN, latent_dim=VAE_LATENT_DIM, hidden_dim=VAE_HIDDEN_DIM, 
                          emb_dim=VAE_EMB_DIM, img_size=VAE_IMG_SIZE).to(vae_device)
gan_model_right = GANomaly(seq_len=VAE_SEQ_LEN, latent_dim=VAE_LATENT_DIM, hidden_dim=VAE_HIDDEN_DIM, 
                           emb_dim=VAE_EMB_DIM, img_size=VAE_IMG_SIZE).to(vae_device)

gan_weights_left = Path("gan_anomaly_weights_left.pt")
gan_weights_right = Path("gan_anomaly_weights_right.pt")

if gan_weights_left.exists():
    gan_model_left.load_state_dict(torch.load(gan_weights_left, map_location=vae_device))
    print("✓ Loaded GANomaly (left) from disk")
else:
    print("Training GANomaly (left)...")
    train_ganomaly(gan_model_left, train_left_loader_merged, vae_device, epochs=10, lr_gen=1e-4, lr_disc=5e-5)
    torch.save(gan_model_left.state_dict(), gan_weights_left)
    print(f"✓ Saved to {gan_weights_left}")

if gan_weights_right.exists():
    gan_model_right.load_state_dict(torch.load(gan_weights_right, map_location=vae_device))
    print("✓ Loaded GANomaly (right) from disk")
else:
    print("Training GANomaly (right)...")
    train_ganomaly(gan_model_right, train_right_loader_merged, vae_device, epochs=10, lr_gen=1e-4, lr_disc=5e-5)
    torch.save(gan_model_right.state_dict(), gan_weights_right)
    print(f"✓ Saved to {gan_weights_right}")

gan_model_left.eval()
gan_model_right.eval()
print("✓ GANomaly training complete")

Left: normal=1800 + synthetic=1800 = 3600 total
Right: normal=1800 + synthetic=1800 = 3600 total
Initializing GANomaly models...
✓ Loaded GANomaly (left) from disk
✓ Loaded GANomaly (right) from disk
✓ GANomaly training complete


In [5]:
# Cell 5d: Calibrate VAE & GANomaly thresholds on normal data
from vae_anomaly_module import calibrate_threshold, sequence_anomaly_score
from vae_anomaly_module import ganomaly_anomaly_score

print("Calibrating anomaly thresholds...")

# VAE thresholds (on purely normal data)
threshold_left_vae, _ = calibrate_threshold(
    vae_model_left, make_loader(normal_left_seqs, batch_size=VAE_BATCH_SIZE), 
    vae_device, percentile=98.5, kl_weight=VAE_TRAIN_KL_WEIGHT
)
threshold_right_vae, _ = calibrate_threshold(
    vae_model_right, make_loader(normal_right_seqs, batch_size=VAE_BATCH_SIZE), 
    vae_device, percentile=98.5, kl_weight=VAE_TRAIN_KL_WEIGHT
)

# GANomaly thresholds
def calibrate_ganomaly_threshold(model, normal_loader, device, percentile=98.5):
    scores = []
    for (x,) in normal_loader:
        s = ganomaly_anomaly_score(model, x, device)
        scores.append(s)
    all_scores = np.concatenate(scores) if scores else np.array([], dtype=np.float32)
    if all_scores.size == 0:
        raise RuntimeError("No scores for GANomaly calibration")
    return float(np.percentile(all_scores, percentile)), all_scores

threshold_left_gan, _ = calibrate_ganomaly_threshold(
    gan_model_left, make_loader(normal_left_seqs, batch_size=VAE_BATCH_SIZE), vae_device
)
threshold_right_gan, _ = calibrate_ganomaly_threshold(
    gan_model_right, make_loader(normal_right_seqs, batch_size=VAE_BATCH_SIZE), vae_device
)

print(f"✓ VAE thresholds:      L={threshold_left_vae:.6f}, R={threshold_right_vae:.6f}")
print(f"✓ GANomaly thresholds: L={threshold_left_gan:.6f}, R={threshold_right_gan:.6f}")
print("✓ Threshold calibration complete")

Calibrating anomaly thresholds...
✓ VAE thresholds:      L=0.010521, R=0.078155
✓ GANomaly thresholds: L=0.045682, R=0.147530
✓ Threshold calibration complete


In [10]:
# Cell: Build training loaders (exclude synthetic reverse samples)
print("Building training loaders (excluding synthetic reverse samples).")

import numpy as np
import pathlib

def _exclude_wrong_way(arr):
    # If already a numpy array, leave as-is
    if isinstance(arr, np.ndarray):
        return arr
    try:
        arr = list(arr) if arr is not None else []
    except Exception:
        arr = []
    # If elements look like file paths, filter by path string
    if arr and all(isinstance(p, (str, pathlib.Path)) for p in arr):
        return np.array([p for p in arr if 'vae_wrong_way_samples' not in str(p)])
    # Otherwise assume list of numeric sequences -> convert to numpy array
    try:
        return np.array(arr)
    except Exception:
        return np.array([])

def _ensure_numeric(arr):
    # Ensure we return a numeric numpy array suitable for make_loader
    if not isinstance(arr, np.ndarray):
        arr = np.array(arr)
    if arr.dtype == object:
        # try to stack elements (common when elements are arrays)
        try:
            return np.stack(arr)
        except Exception:
            # fallback to empty numeric array
            return np.array([])
    return arr

# Resolve left merged training set (fall back sensibly if names differ)
train_left_merged = globals().get('train_left_merged', None)
if train_left_merged is None:
    train_left_merged = globals().get('train_left', globals().get('normal_left_seqs', []))
train_left_merged = _exclude_wrong_way(train_left_merged)
train_left_merged = _ensure_numeric(train_left_merged)

# Resolve right merged training set
train_right_merged = globals().get('train_right_merged', None)
if train_right_merged is None:
    train_right_merged = globals().get('train_right', globals().get('normal_right_seqs', []))
train_right_merged = _exclude_wrong_way(train_right_merged)
train_right_merged = _ensure_numeric(train_right_merged)

# Create loaders for VAE training (no reversed/synthetic samples included)
train_left_loader_merged = make_loader(train_left_merged, batch_size=VAE_BATCH_SIZE, shuffle=True)
train_right_loader_merged = make_loader(train_right_merged, batch_size=VAE_BATCH_SIZE, shuffle=True)

print("GANomaly training complete.")


Building training loaders (excluding synthetic reverse samples).
GANomaly training complete.


In [11]:
# Cell 10: VAE vs GANomaly Separate Comparison with Bounding Box Visualization
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import deque
import torch

from vae_anomaly_module import ganomaly_anomaly_score

# --- 1) DETECTION CONFIG ---
PERCENTILE_LEFT = 98.5
PERCENTILE_RIGHT = 98.5
KL_WEIGHT = 0.05

# --- 1b) VIDEO OUTPUT CONFIG ---
OUTPUT_VIDEO = "vae_gan_comparison_anomalies.mp4"

# Validate required objects
required = [
    "vae_model_left", "vae_model_right", "vae_device",
    "gan_model_left", "gan_model_right",
    "normal_loader_left", "normal_loader_right",
    "VAE_LEFT_ROI", "VAE_RIGHT_ROI",
    "vae_timeline_source", "VAE_SEQ_LEN", "VAE_FRAME_STEP", "VAE_SEQ_STRIDE", "VAE_IMG_SIZE",
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError("Missing required objects: " + ", ".join(missing))

# --- 2) CALIBRATE THRESHOLDS ---
print("Calibrating thresholds for both models...")

# VAE thresholds
from vae_anomaly_module import calibrate_threshold
threshold_left_vae, _ = calibrate_threshold(
    vae_model_left, normal_loader_left, vae_device, percentile=PERCENTILE_LEFT, kl_weight=KL_WEIGHT
)
threshold_right_vae, _ = calibrate_threshold(
    vae_model_right, normal_loader_right, vae_device, percentile=PERCENTILE_RIGHT, kl_weight=KL_WEIGHT
)

# GANomaly thresholds
def calibrate_ganomaly_threshold(model, loader, device, percentile=98.5):
    scores = []
    with torch.no_grad():
        for (x,) in loader:
            x = x.to(device)
            s = ganomaly_anomaly_score(model, x, device)
            scores.extend(s)
    all_scores = np.array(scores, dtype=np.float32)
    return float(np.percentile(all_scores, percentile))

threshold_left_gan = calibrate_ganomaly_threshold(gan_model_left, normal_loader_left, vae_device)
threshold_right_gan = calibrate_ganomaly_threshold(gan_model_right, normal_loader_right, vae_device)

THR_LEFT_VAE = threshold_left_vae
THR_RIGHT_VAE = threshold_right_vae
THR_LEFT_GAN = threshold_left_gan
THR_RIGHT_GAN = threshold_right_gan

print(f"VAE thresholds:      L={THR_LEFT_VAE:.6f}, R={THR_RIGHT_VAE:.6f}")
print(f"GANomaly thresholds: L={THR_LEFT_GAN:.6f}, R={THR_RIGHT_GAN:.6f}")

# --- 3) OPEN VIDEO & SETUP ---
timeline_source = str(vae_timeline_source)
cap = cv2.VideoCapture(timeline_source)
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {timeline_source}")

fps = cap.get(cv2.CAP_PROP_FPS) or 30
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Extract ROI coordinates
x1_l, y1_l, x2_l, y2_l = VAE_LEFT_ROI
x1_r, y1_r, x2_r, y2_r = VAE_RIGHT_ROI

# Video writer
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (frame_w * 2, frame_h))

# Sequence buffers
seq_len = VAE_SEQ_LEN
seq_buffer_left = deque(maxlen=seq_len)
seq_buffer_right = deque(maxlen=seq_len)
frame_counter = 0

vae_left_scores_all = []
vae_right_scores_all = []
gan_left_scores_all = []
gan_right_scores_all = []
alert_rows = []

def compute_scores(seq_left, seq_right, frame_idx):
    """Compute VAE and GAN scores for a sequence.
    
    Args:
        seq_left: numpy array shape (B, T, C, H, W)
        seq_right: numpy array shape (B, T, C, H, W)
        frame_idx: frame index (unused, for logging)
    
    Returns:
        tuple: (vae_score_l, vae_score_r, gan_score_l, gan_score_r)
    """
    seq_left_t = torch.from_numpy(seq_left).float().to(vae_device)
    seq_right_t = torch.from_numpy(seq_right).float().to(vae_device)
    
    with torch.no_grad():
        # VAE scoring
        vae_model_left.eval()
        vae_model_right.eval()
        vae_recon_l, vae_mu_l, vae_logvar_l = vae_model_left(seq_left_t)
        vae_recon_r, vae_mu_r, vae_logvar_r = vae_model_right(seq_right_t)
        
        vae_score_l = torch.mean((seq_left_t - vae_recon_l) ** 2).item()
        vae_score_r = torch.mean((seq_right_t - vae_recon_r) ** 2).item()
        
        kl_l = -0.5 * torch.mean(1 + vae_logvar_l - vae_mu_l**2 - vae_logvar_l.exp())
        kl_r = -0.5 * torch.mean(1 + vae_logvar_r - vae_mu_r**2 - vae_logvar_r.exp())
        
        vae_score_l += KL_WEIGHT * kl_l.item()
        vae_score_r += KL_WEIGHT * kl_r.item()
        
        # GANomaly scoring
        gan_model_left.eval()
        gan_model_right.eval()
        gan_score_l = ganomaly_anomaly_score(gan_model_left, seq_left_t, vae_device)[0]
        gan_score_r = ganomaly_anomaly_score(gan_model_right, seq_right_t, vae_device)[0]
    
    return vae_score_l, vae_score_r, gan_score_l, gan_score_r

print("Processing video with VAE + GANomaly...")
while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    frame_counter += 1
    
    # Extract ROI crops
    left_crop = frame[y1_l:y2_l, x1_l:x2_l].astype(np.float32) / 255.0
    right_crop = frame[y1_r:y2_r, x1_r:x2_r].astype(np.float32) / 255.0
    
    left_gray = cv2.cvtColor((left_crop * 255).astype(np.uint8), cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
    right_gray = cv2.cvtColor((right_crop * 255).astype(np.uint8), cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
    
    left_gray_resized = cv2.resize(left_gray, (VAE_IMG_SIZE, VAE_IMG_SIZE))
    right_gray_resized = cv2.resize(right_gray, (VAE_IMG_SIZE, VAE_IMG_SIZE))
    
    seq_buffer_left.append(left_gray_resized[np.newaxis, :, :])
    seq_buffer_right.append(right_gray_resized[np.newaxis, :, :])
    
    # Process when we have a full sequence
    if len(seq_buffer_left) == seq_len:
        # Stack: (seq_len, 1, H, W) then add batch dim: (1, seq_len, 1, H, W)
        seq_left_np = np.array(list(seq_buffer_left), dtype=np.float32)[np.newaxis, ...]
        seq_right_np = np.array(list(seq_buffer_right), dtype=np.float32)[np.newaxis, ...]
        
        vae_score_l, vae_score_r, gan_score_l, gan_score_r = compute_scores(seq_left_np, seq_right_np, frame_counter)
        
        vae_left_scores_all.append(vae_score_l)
        vae_right_scores_all.append(vae_score_r)
        gan_left_scores_all.append(gan_score_l)
        gan_right_scores_all.append(gan_score_r)
        
        # Determine anomaly status
        vae_left_anom = vae_score_l > THR_LEFT_VAE
        vae_right_anom = vae_score_r > THR_RIGHT_VAE
        gan_left_anom = gan_score_l > THR_LEFT_GAN
        gan_right_anom = gan_score_r > THR_RIGHT_GAN
        
        alert_rows.append({
            'frame': frame_counter,
            'vae_left_score': vae_score_l,
            'vae_right_score': vae_score_r,
            'gan_left_score': gan_score_l,
            'gan_right_score': gan_score_r,
            'vae_left_anom': int(vae_left_anom),
            'vae_right_anom': int(vae_right_anom),
            'gan_left_anom': int(gan_left_anom),
            'gan_right_anom': int(gan_right_anom),
        })
        
        # Create side-by-side frames with anomaly boxes
        frame_vae = frame.copy()
        frame_gan = frame.copy()
        
        # VAE bounding boxes (orange)
        if vae_left_anom:
            cv2.rectangle(frame_vae, (int(x1_l), int(y1_l)), (int(x2_l), int(y2_l)), (0, 165, 255), 3)
            cv2.putText(frame_vae, f"VAE L: {vae_score_l:.4f}", (int(x1_l) + 5, int(y1_l) - 5),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2)
        if vae_right_anom:
            cv2.rectangle(frame_vae, (int(x1_r), int(y1_r)), (int(x2_r), int(y2_r)), (0, 165, 255), 3)
            cv2.putText(frame_vae, f"VAE R: {vae_score_r:.4f}", (int(x1_r) + 5, int(y1_r) - 5),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2)
        
        # GANomaly bounding boxes (green)
        if gan_left_anom:
            cv2.rectangle(frame_gan, (int(x1_l), int(y1_l)), (int(x2_l), int(y2_l)), (0, 255, 0), 3)
            cv2.putText(frame_gan, f"GAN L: {gan_score_l:.4f}", (int(x1_l) + 5, int(y1_l) - 5),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        if gan_right_anom:
            cv2.rectangle(frame_gan, (int(x1_r), int(y1_r)), (int(x2_r), int(y2_r)), (0, 255, 0), 3)
            cv2.putText(frame_gan, f"GAN R: {gan_score_r:.4f}", (int(x1_r) + 5, int(y1_r) - 5),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        
        # Add labels
        cv2.putText(frame_vae, "VAE", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 165, 255), 2)
        cv2.putText(frame_gan, "GANomaly", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 2)
        
        # Combine side-by-side
        combined = np.hstack([frame_vae, frame_gan])
        out.write(combined)
    
    if frame_counter % max(1, total_frames // 10) == 0 and total_frames > 0:
        pct = 100.0 * frame_counter / total_frames
        print(f"  {pct:.1f}% ({frame_counter}/{total_frames} frames)")

cap.release()
out.release()

# --- 4) EXPORT RESULTS ---
alert_df = pd.DataFrame(alert_rows)
alert_df.to_csv("vae_gan_comparison_scores.csv", index=False)
print(f"\n✓ Saved comparison scores to vae_gan_comparison_scores.csv")
print(f"✓ Saved side-by-side video to {OUTPUT_VIDEO}")

# --- 5) SUMMARY STATISTICS ---
print(f"\n--- VAE Results ---")
print(f"Left  - Mean: {np.mean(vae_left_scores_all):.6f}, Threshold: {THR_LEFT_VAE:.6f}, Anomalies: {sum(1 for s in vae_left_scores_all if s > THR_LEFT_VAE)}")
print(f"Right - Mean: {np.mean(vae_right_scores_all):.6f}, Threshold: {THR_RIGHT_VAE:.6f}, Anomalies: {sum(1 for s in vae_right_scores_all if s > THR_RIGHT_VAE)}")

print(f"\n--- GANomaly Results ---")
print(f"Left  - Mean: {np.mean(gan_left_scores_all):.6f}, Threshold: {THR_LEFT_GAN:.6f}, Anomalies: {sum(1 for s in gan_left_scores_all if s > THR_LEFT_GAN)}")
print(f"Right - Mean: {np.mean(gan_right_scores_all):.6f}, Threshold: {THR_RIGHT_GAN:.6f}, Anomalies: {sum(1 for s in gan_right_scores_all if s > THR_RIGHT_GAN)}")

# Display sample comparisons
print(f"\n--- Sample Detections (First 20) ---")
display(alert_df.head(20))

Calibrating thresholds for both models...
VAE thresholds:      L=0.010503, R=0.077897
GANomaly thresholds: L=0.045682, R=0.147530
Processing video with VAE + GANomaly...
  10.0% (389/3899 frames)
  20.0% (778/3899 frames)
  29.9% (1167/3899 frames)
  39.9% (1556/3899 frames)
  49.9% (1945/3899 frames)
  59.9% (2334/3899 frames)
  69.8% (2723/3899 frames)
  79.8% (3112/3899 frames)
  89.8% (3501/3899 frames)
  99.8% (3890/3899 frames)

✓ Saved comparison scores to vae_gan_comparison_scores.csv
✓ Saved side-by-side video to vae_gan_comparison_anomalies.mp4

--- VAE Results ---
Left  - Mean: 0.023170, Threshold: 0.010503, Anomalies: 3884
Right - Mean: 0.017777, Threshold: 0.077897, Anomalies: 0

--- GANomaly Results ---
Left  - Mean: 0.057457, Threshold: 0.045682, Anomalies: 3884
Right - Mean: 0.088762, Threshold: 0.147530, Anomalies: 0

--- Sample Detections (First 20) ---


,frame,vae_left_score,vae_right_score,gan_left_score,gan_right_score,vae_left_anom,vae_right_anom,gan_left_anom,gan_right_anom
0,16,0.024907,0.017443,0.061025,0.091472,1,0,1,0
1,17,0.024810,0.017110,0.060836,0.090650,1,0,1,0
2,18,0.024632,0.016946,0.060645,0.090338,1,0,1,0
3,19,0.024463,0.016918,0.060479,0.090213,1,0,1,0
4,20,0.024453,0.016960,0.060316,0.090244,1,0,1,0
5,21,0.024330,0.016988,0.060164,0.090229,1,0,1,0
6,22,0.024230,0.016997,0.060005,0.090237,1,0,1,0
7,23,0.024089,0.017032,0.059871,0.090195,1,0,1,0
8,24,0.024085,0.017014,0.059758,0.090194,1,0,1,0
9,25,0.024003,0.017063,0.059664,0.090088,1,0,1,0
